# 🚀 assist — Google Colab T4 GPU Ollama Runner & Benchmark

`assist`의 한국어 3줄 요약 및 RAG 모델 비교 검증(이슈 #19)을 위한 Colab GPU 실행기입니다.

### 📌 사용 방법
1. 상단 메뉴에서 **런타임 > 런타임 유형 변경 > T4 GPU**를 선택합니다.
2. 아래 셀들을 순서대로 실행합니다.
3. 셀 2에서 출력되는 `https://xxxx.trycloudflare.com` URL을 복사하여 로컬 또는 GCP `assist-api`의 `OLLAMA_BASE_URL`로 지정할 수 있습니다.

In [ ]:
# 1. GPU 사양 확인 및 Ollama 설치 & 모델 다운로드
!nvidia-smi
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
print("Starting Ollama daemon in background...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

print("Pulling Model A (gemma3:12b) & Model C (hermes3:8b)...")
!ollama pull gemma3:12b
!ollama pull hermes3:8b
print("✅ Models ready!")

In [ ]:
# 2. Cloudflare Quick Tunnel 실행 (외부 HTTPS URL 생성)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

import re
print("Starting Cloudflare Quick Tunnel to port 11434...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stderr=subprocess.PIPE,
    text=True,
)

tunnel_url = None
for line in tunnel_proc.stderr:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        print(f"\n🌟 외부 접속용 OLLAMA_BASE_URL = {tunnel_url}\n")
        break

if not tunnel_url:
    print("Tunnel URL not captured yet. Check logs.")

In [ ]:
# 3. 동일 프롬프트 대상 모델 A vs 모델 C 한국어 3줄 요약 비교 벤치마크
import requests, time

prompt = """다음 기술 동향을 개발자를 위해 핵심만 한국어 3줄 요약해줘:
- Next.js 16: 내장 DevTools MCP 서버 탑재 및 Cache Components 정식화
- NestJS 12: ESM 전면 전환 및 Standard Schema(Zod, Valibot) 지원
- Claude Code: 서브에이전트 오케스트레이션 및 review-army 규율 연동
"""

models = ["gemma3:12b", "hermes3:8b"]
for m in models:
    print(f"\n--- Testing {m} ---")
    t0 = time.time()
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": m, "prompt": prompt, "stream": False},
        timeout=120,
    ).json()
    elapsed = time.time() - t0
    tokens = res.get("eval_count", 0)
    speed = tokens / (res.get("eval_duration", 1) / 1e9)
    print(f"⚡ 소요 시간: {elapsed:.2f}초 | 속도: {speed:.1f} tokens/s")
    print("📄 응답 결과:")
    print(res.get("response", "").strip())

In [ ]:
# 4. (선택) assist-api의 POST /brief/compare 엔드포인트 연동
# assist-api가 실행 중인 경우 아래 주소를 설정하여 POST /brief/compare를 호출할 수 있습니다.
import os
ASSIST_API_URL = os.environ.get("ASSIST_API_URL", "http://localhost:8000")
try:
    resp = requests.post(
        f"{ASSIST_API_URL}/brief/compare",
        json={"models": ["A", "C"]},
        timeout=180,
    )
    if resp.status_code == 200:
        print("POST /brief/compare Result:")
        print(resp.json())
    else:
        print(f"Status: {resp.status_code}, Body: {resp.text}")
except Exception as e:
    print(f"Note: {e}")